# Exhash Tests

This notebook tests the **exhash** (line-hash-addressed editor) module, which provides verified text editing using content-hash addresses.

## Key Concepts

- **lnhash address**: `lineno|hash|` — A 4-char hex hash derived from line content, used to verify edits target the correct line
- **lnhashview**: Shows all lines with their hash addresses for reference
- **exhash commands**: Edit operations addressed by `lineno|hash|command` (like `ex` editor but hash-verified)

## Functions

### Core (`exhash` package)
- `lnhashview(text)` — Show lines with hash addresses
- `exhash(text, cmds)` — Apply hash-addressed edit commands, returns dict with `lines`, `hashes`, `modified`, `deleted`
- `lnhash(lineno, line)` — Get hash address for a specific line
- `line_hash(line)` — Get just the 4-char hash for a line
- `exhash_result(results)` — Format modified lines from result dicts

### DialogHelper integration (`dialoghelper.exhash`)
- `msg_lnhashview(id)` — Show hash-addressed lines of a notebook cell
- `msg_exhash(id, cmds)` — Apply exhash commands to a cell's content
- `file_lnhashview(path)` — Show hash-addressed lines of a file
- `file_exhash(path, cmds)` — Apply exhash commands to a file

**Run each cell in order.**

In [ ]:
# Cell 1: Imports
from exhash import lnhashview, exhash, lnhash, line_hash, exhash_result

print('Exhash imports ready!')

In [ ]:
# Cell 2: lnhashview — show lines with hash addresses
text = """The quick brown fox jumps over the lazy dog.
Pack my box with five dozen liquor jugs.
How vexingly quick daft zebras jump.
The five boxing wizards jump quickly."""

print('=== lnhashview ===')
for line in lnhashview(text):
    print(line)

print(f'\nFormat: lineno|hash|  content')
print(f'Hash is a 4-char hex digest of the line content')

In [ ]:
# Cell 3: lnhash and line_hash — get addresses for specific lines
line = 'The quick brown fox jumps over the lazy dog.'

# line_hash returns just the 4-char hash
h = line_hash(line)
print(f'line_hash: {h!r}')

# lnhash returns the full address: lineno|hash|
addr = lnhash(1, line)
print(f'lnhash(1, ...): {addr!r}')

addr3 = lnhash(3, 'How vexingly quick daft zebras jump.')
print(f'lnhash(3, ...): {addr3!r}')

In [ ]:
# Cell 4: exhash — substitute command (s/pat/rep/)
text = """Alpha line.
Bravo line.
Charlie line.
Delta line."""

print('=== Original ===')
for l in lnhashview(text): print(l)

# Compute hash dynamically for line 2
lines = text.split('\n')
h2 = line_hash(lines[1])

# Substitute 'Bravo' with 'BRAVO' on line 2
result = exhash(text, [f'2|{h2}|s/Bravo/BRAVO/'])
print('\n=== After s/Bravo/BRAVO/ on line 2 ===')
for l in lnhashview('\n'.join(result['lines'])): print(l)
print(f'modified: {result["modified"]}')

In [ ]:
# Cell 5: exhash — delete command (d)
text = """Alpha line.
Bravo line.
Charlie line.
Delta line."""

print('=== Original ===')
for l in lnhashview(text): print(l)

# Compute hash dynamically for line 2
lines = text.split('\n')
h2 = line_hash(lines[1])

# Delete line 2
result = exhash(text, [f'2|{h2}|d'])
print('\n=== After deleting line 2 ===')
for l in lnhashview('\n'.join(result['lines'])): print(l)
print(f'deleted: {result["deleted"]}')

In [ ]:
# Cell 6: exhash — insert (i) and append (a) commands
text = """Alpha line.
Bravo line.
Charlie line."""

print('=== Original ===')
for l in lnhashview(text): print(l)

# Compute hashes dynamically
lines = text.split('\n')
h1 = line_hash(lines[0])
h2 = line_hash(lines[1])

# Insert before line 2 (text block follows after newline)
result = exhash(text, [f'2|{h2}|i\nINSERTED BEFORE BRAVO'])
print('\n=== After insert before line 2 ===')
for l in lnhashview('\n'.join(result['lines'])): print(l)

# Append after line 1
result2 = exhash(text, [f'1|{h1}|a\nAPPENDED AFTER ALPHA'])
print('\n=== After append after line 1 ===')
for l in lnhashview('\n'.join(result2['lines'])): print(l)

In [ ]:
# Cell 7: exhash — change command (c) — replace entire line(s)
text = """Alpha line.
Bravo line.
Charlie line."""

print('=== Original ===')
for l in lnhashview(text): print(l)

# Compute hash dynamically for line 2
lines = text.split('\n')
h2 = line_hash(lines[1])

# Change line 2 to completely new content
result = exhash(text, [f'2|{h2}|c\nCOMPLETELY NEW LINE TWO'])
print('\n=== After change line 2 ===')
for l in lnhashview('\n'.join(result['lines'])): print(l)
print(f'modified: {result["modified"]}')

In [ ]:
# Cell 8: exhash — indent (>) and dedent (<) commands
text = """def hello():
print('world')
return True"""

print('=== Original ===')
for l in lnhashview(text): print(l)

# Get hashes for lines 2 and 3
lines = text.split('\n')
h2 = line_hash(lines[1])
h3 = line_hash(lines[2])

# Indent lines 2-3 (range address)
result = exhash(text, [f'2|{h2}|,3|{h3}|>'])
print('\n=== After indenting lines 2-3 ===')
for l in lnhashview('\n'.join(result['lines'])): print(l)

In [ ]:
# Cell 9: exhash — multiple commands in one call
text = """Line A
Line B
Line C
Line D
Line E"""

print('=== Original ===')
for l in lnhashview(text): print(l)

# Get hashes
lines = text.split('\n')
hashes = {i+1: line_hash(l) for i, l in enumerate(lines)}
print(f'Hashes: {hashes}')

# Apply multiple commands: substitute on line 1, delete line 3
# Note: hashes are verified once before all commands run
result = exhash(text, [
    f'1|{hashes[1]}|s/A/ALPHA/',
    f'3|{hashes[3]}|d'
])
print('\n=== After sub line 1 + delete line 3 ===')
for l in lnhashview('\n'.join(result['lines'])): print(l)
print(f'modified: {result["modified"]}, deleted: {result["deleted"]}')

In [ ]:
# Cell 10: exhash_result — format modified lines from results
text = """First line.
Second line.
Third line."""

result = exhash(text, [f'2|{line_hash("Second line.")}|s/Second/SECOND/'])
print('=== exhash_result (shows only modified lines) ===')
print(exhash_result([result]))
print(f'\n(Only line {result["modified"]} shown — the modified one)')

In [ ]:
# Cell 11: File operations — lnhashview and exhash on files
# Note: dialoghelper's file_lnhashview/file_exhash have a missing Path import,
# so we use the core exhash functions with pathlib directly.
from pathlib import Path
import tempfile, os

# Create a temp file
tmp = Path(tempfile.mktemp(suffix='.txt'))
tmp.write_text('Alpha line.\nBravo line.\nCharlie line.\nDelta line.\n')

# View with hashes
print('=== File lnhashview ===')
for l in lnhashview(tmp.read_text()): print(l)

# Edit: substitute on line 2
content = tmp.read_text()
result = exhash(content, [f'2|{line_hash("Bravo line.")}|s/Bravo/BRAVO/'])
new_content = '\n'.join(result['lines'])
tmp.write_text(new_content)

print('\n=== After editing file ===')
for l in lnhashview(tmp.read_text()): print(l)

# Cleanup
tmp.unlink()
print('\nTemp file cleaned up')

In [ ]:
# Cell 12: Hash verification — wrong hash is rejected
text = """Line one.
Line two.
Line three."""

print('=== Original ===')
for l in lnhashview(text): print(l)

# Try with wrong hash — should raise an error
try:
    exhash(text, ['2|XXXX|d'])
    print('ERROR: Should have raised!')
except Exception as e:
    print(f'\nCorrectly rejected wrong hash: {type(e).__name__}: {e}')
    print('This is the key safety feature — edits are verified against content hashes')

In [ ]:
# Cell 13: Global command (g/pat/cmd) — apply command to matching lines
text = """apple pie
banana split
apple sauce
cherry tart
apple crisp"""

print('=== Original ===')
for l in lnhashview(text): print(l)

# Get hash for line 1 (needed for the global address)
lines = text.split('\n')
h1 = line_hash(lines[0])
h5 = line_hash(lines[4])

# Global substitute: uppercase 'apple' on all lines that contain it
result = exhash(text, [f'1|{h1}|,5|{h5}|g/apple/s/apple/APPLE/'])
print('\n=== After g/apple/s/apple/APPLE/ ===')
for l in lnhashview('\n'.join(result['lines'])): print(l)
print(f'modified: {result["modified"]}')

## Summary

If all cells ran successfully, you've verified:

- **lnhashview()** — Display lines with `lineno|hash|` addresses
- **lnhash() / line_hash()** — Get hash addresses for specific lines
- **exhash()** — Apply verified edit commands:
  - `s/pat/rep/` — Substitute (regex)
  - `d` — Delete line(s)
  - `i` / `a` — Insert before / Append after
  - `c` — Change (replace) line(s)
  - `>` / `<` — Indent / Dedent
  - `g/pat/cmd` — Global (apply command to matching lines)
- **exhash_result()** — Format only modified lines from results
- **File editing** — Read/edit/write files using exhash
- **Hash verification** — Wrong hashes are rejected (safety feature)
- **Multiple commands** — Apply several edits in one call (hashes verified once upfront)

### How It Works

```
lnhashview(text)          # Get hash addresses for all lines
  → ['1|a3f2|  Line one', '2|b1c3|  Line two', ...]

exhash(text, cmds)        # Apply verified edits
  → {'lines': [...], 'hashes': [...], 'modified': [2], 'deleted': []}

# Commands use lnhash addresses:
#   2|b1c3|s/old/new/     Substitute on line 2 (verified by hash b1c3)
#   2|b1c3|d              Delete line 2
#   2|b1c3|c\nNew text    Change/replace line 2
#   2|b1c3|i\nNew line    Insert before line 2
#   2|b1c3|a\nNew line    Append after line 2
#   2|b1c3|,4|d5e6|>      Indent lines 2-4
```

### Why Hash Addresses?

Hash addresses prevent **stale edit** errors — if line content changes between viewing and editing, the hash won't match and the edit is rejected. This is critical for LLM tool calling where the model may reference line numbers from an earlier view.

### Requirements

- `exhash` package (installed as dependency of dialoghelper)
- `dialoghelper` package (for `msg_lnhashview`, `msg_exhash` cell editing)